# Chatbot Agent

The goal is to build a chatbot that understands what a user wants (the intent of the message), answers with a prepared reply, and hands over to a language model when the message is not about anything it was built for. The training data is the small intent file of the reference (`intents.json`: 8 intents such as greeting, thanks, help, create an account and complaint, with 3 to 7 example messages each).

Reference solution: [Create a Chatbot with Python and Machine Learning](https://amanxai.com/2020/11/01/chatbot-with-machine-learning-and-python/) (Aman Kharwal). It trains a small neural network (embedding layer, average pooling and dense layers) for 500 epochs on the 33 example messages, and then answers every message with a random reply of the intent with the highest probability. It never tests the network on messages that were not used for training, and the bot always answers, even to a message that has nothing to do with its intents.

This notebook tests the reference network on new messages, compares it with two simpler models (TF-IDF with Logistic Regression, and the nearest training message), adds a confidence threshold so that the bot can say "this is not for me", and hands those messages to a local language model (Llama 3.2, run by Ollama; no API key or online service is used). The agent is a fixed workflow: classify, then answer with a prepared reply or ask the language model.

In [2]:
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
import os
os.chdir('/mnt/c/Users/myama/OneDrive/Belgeler/ai-1/Assignments/14-Specialize in Data Science/AI Agents/chatbot_agent')
tf.keras.utils.set_random_seed(42)

## Data Loading

In [3]:
intents = json.load(open("intents.json"))["intents"]
training = pd.DataFrame([(pattern, intent["tag"]) for intent in intents for pattern in intent["patterns"]], columns=["message", "intent"])
print(len(training), "training messages,", training["intent"].nunique(), "intents")
training["intent"].value_counts()

33 training messages, 8 intents


intent
help             7
greeting         5
createaccount    5
thanks           4
goodbye          3
about            3
name             3
complaint        3
Name: count, dtype: int64

## Test Messages

The reference has no test data, so it is written here. For each of the 8 intents, 10 new messages with other words than the training messages were written (80 messages), and 50 messages that have nothing to do with any of the intents (weather, jokes, geography, recipes and so on). Half of them are used to choose the confidence threshold (the development set) and the other half to measure the result (the test set), so that the threshold is not chosen on the test messages. These messages were written for this notebook and are few, so the scores below are only a rough guide.

In [4]:
in_scope = {
    "greeting": ["Hello there", "Good morning", "Hey, how are you", "Hi bot", "Anybody home?", "Greetings", "Yo", "Hello, is somebody there?", "Hiya", "Hey there"],
    "goodbye": ["Goodbye for now", "Talk to you later", "See you soon", "I have to go, bye", "Farewell", "Catch you later", "Bye bye", "Time to leave, goodbye", "See ya", "Take care, bye"],
    "thanks": ["Thanks a lot", "Thank you so much", "Many thanks", "I appreciate your help", "That was really helpful", "Cheers, thank you", "Thanks, that helps", "Great, thanks", "Much appreciated", "Thank you for your support"],
    "about": ["What kind of bot are you?", "Tell me about yourself", "Are you a robot?", "Who am I talking to?", "What are you exactly?", "Are you a human or a machine?", "Introduce yourself", "What is this assistant?", "Who is this?", "Who made you?"],
    "name": ["How should I address you?", "Can I know your name?", "Do you have a name?", "What do people call you?", "Tell me your name", "May I ask your name?", "What's the name of this bot?", "How do I refer to you?", "Name please", "What name do you go by?"],
    "help": ["Can you assist me?", "I need some assistance", "Would you help me with something?", "Help me please", "I have a problem and need support", "Is there anyone who can help?", "Could you lend me a hand?", "What can you help with?", "I'm stuck, please help", "I want some help"],
    "createaccount": ["How do I sign up?", "I would like to register a new account", "Where can I create an account?", "Steps to open an account", "Please help me make a new account", "I want to sign up for your service", "How can I register?", "Create a new profile for me", "Where do I open an account?", "I don't have an account yet, how do I get one?"],
    "complaint": ["I want to complain about something", "I'd like to report a problem with your service", "I am not happy with the service", "Where can I file a complaint?", "I have a complaint to make", "The service was terrible and I want to complain", "I need to lodge a complaint", "Something went wrong and I am unhappy", "How do I make a complaint?", "I want to complain about my order"],
}
out_of_scope = ["What is the weather like today?", "Tell me a joke", "What is the capital of France?", "How do I bake a chocolate cake?", "Who won the football match yesterday?",
    "What time is it in Tokyo?", "Recommend a good movie", "How far is the moon from the earth?", "Translate good morning into Spanish", "What is 15 times 12?", "Play some music",
    "Who is the president of the United States?", "How do I fix a flat tire?", "What is the meaning of life?", "Give me a recipe for pasta", "How tall is Mount Everest?",
    "Book me a flight to Paris", "What is the stock price of Apple?", "Tell me a fun fact about cats", "Write a poem about the sea", "How do I lose weight fast?",
    "What is the best programming language?", "When does the next train leave?", "Explain quantum computing", "Where is the nearest pharmacy?", "What year did World War II end?",
    "How many legs does a spider have?", "Can you sing a song?", "Suggest a name for my dog", "What is the boiling point of water?", "How do I learn to play the guitar?",
    "What are the symptoms of the flu?", "Tell me about the history of Rome", "Is it going to rain tomorrow?", "Who painted the Mona Lisa?", "Convert 100 dollars to euros",
    "What is the largest animal on earth?", "How do I change my cars oil?", "What is your favourite colour?", "Do you like pizza?", "How many days are in a leap year?", "Give me a workout plan",
    "What is the speed of light?", "Which planet is the hottest?", "How do I tie a tie?", "Tell me a bedtime story", "What is machine learning?", "Where can I buy cheap shoes?",
    "What is the population of India?", "How old is the universe?"]

in_dev = pd.DataFrame([(m, tag) for tag, messages in in_scope.items() for m in messages[:5]], columns=["message", "intent"])
in_test = pd.DataFrame([(m, tag) for tag, messages in in_scope.items() for m in messages[5:]], columns=["message", "intent"])
out_dev, out_test = out_of_scope[:25], out_of_scope[25:]
print("in-scope messages: development", len(in_dev), "| test", len(in_test), "| out-of-scope: development", len(out_dev), "| test", len(out_test))

in-scope messages: development 40 | test 40 | out-of-scope: development 25 | test 25


## Reference Reproduction

The reference network (vocabulary of 1,000 words, messages cut or padded to 20 words, an embedding of 16 values, average pooling, two dense layers of 16 neurons) is trained for 500 epochs on the 33 messages.

In [5]:
tokenizer = Tokenizer(num_words=1000, oov_token="<OOV>")
tokenizer.fit_on_texts(training["message"])
intent_names = sorted(training["intent"].unique())
to_index = {name: i for i, name in enumerate(intent_names)}

def encode(messages):
    return pad_sequences(tokenizer.texts_to_sequences(messages), truncating="post", maxlen=20)

network = Sequential([Embedding(1000, 16), GlobalAveragePooling1D(), Dense(16, activation="relu"), Dense(16, activation="relu"), Dense(len(intent_names), activation="softmax")])
network.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
history = network.fit(encode(training["message"]), training["intent"].map(to_index).values, epochs=500, verbose=0)
print("accuracy on the training messages:", round(history.history["accuracy"][-1], 3))

I0000 00:00:1790318462.045786   39838 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3536 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9
I0000 00:00:1790318463.349464   39903 service.cc:153] XLA service 0x7841500311c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1790318463.349502   39903 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9 (Driver: 13.3.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.22.0)
I0000 00:00:1790318463.419148   39903 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1790318463.710715   39903 cuda_dnn.cc:461] Loaded cuDNN version 92200
I0000 00:00:1790318463.724603   39903 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1632__.16
I0000 00:00:1790318466.11910

accuracy on the training messages: 1.0


The network reaches an accuracy of 1.0 on its 33 training messages, which is expected after 500 epochs on so few examples: it has memorized them. The reference stops here and reports nothing else. Whether it understands new messages is tested below.

## Models and Confidence

Every model gives an intent and a confidence for a message: the highest probability of the network, the highest probability of the Logistic Regression on TF-IDF vectors of the words (single words and pairs), and the cosine similarity with the most similar training message (the intent of that message is the answer). A message is accepted only if the confidence reaches a threshold; otherwise the bot considers it out of scope. The threshold is chosen on the development messages as the one that maximizes the mean of two shares: the accepted and correct in-scope messages, and the rejected out-of-scope messages.

In [6]:
tfidf = TfidfVectorizer(ngram_range=(1, 2)).fit(training["message"])
logistic = LogisticRegression(C=20, max_iter=1000).fit(tfidf.transform(training["message"]), training["intent"])
train_vectors = tfidf.transform(training["message"])

def network_answer(messages):
    p = network.predict(encode(messages), verbose=0)
    return np.array(intent_names)[p.argmax(axis=1)], p.max(axis=1)

def logistic_answer(messages):
    p = logistic.predict_proba(tfidf.transform(messages))
    return logistic.classes_[p.argmax(axis=1)], p.max(axis=1)

def nearest_answer(messages):
    similarity = cosine_similarity(tfidf.transform(messages), train_vectors)
    return training["intent"].values[similarity.argmax(axis=1)], similarity.max(axis=1)

models = {"Reference network": network_answer, "TF-IDF + Logistic Regression": logistic_answer, "Nearest training message": nearest_answer}

def balanced_score(model, threshold, in_data, out_messages):
    predicted, confidence = model(in_data["message"].tolist())
    _, out_confidence = model(out_messages)
    return ((confidence >= threshold) & (predicted == in_data["intent"].values)).mean() / 2 + (out_confidence < threshold).mean() / 2

thresholds = np.linspace(0, 1, 101)
best_threshold = {name: thresholds[np.argmax([balanced_score(m, th, in_dev, out_dev) for th in thresholds])] for name, m in models.items()}
best_threshold

{'Reference network': np.float64(0.93),
 'TF-IDF + Logistic Regression': np.float64(0.45),
 'Nearest training message': np.float64(0.5700000000000001)}

## Evaluation

In [7]:
def evaluate(model, threshold):
    predicted, confidence = model(in_test["message"].tolist())
    _, out_confidence = model(out_test)
    correct = predicted == in_test["intent"].values
    return {"in-scope accuracy (no threshold)": correct.mean(), "in-scope accepted": (confidence >= threshold).mean(),
            "in-scope accepted and correct": ((confidence >= threshold) & correct).mean(), "out-of-scope rejected": (out_confidence < threshold).mean(),
            "balanced score": (((confidence >= threshold) & correct).mean() + (out_confidence < threshold).mean()) / 2}

results = pd.DataFrame({name: evaluate(m, best_threshold[name]) for name, m in models.items()}).T
results.loc["Reference network, no threshold"] = pd.Series(evaluate(network_answer, 0.0))
results.round(3)

,in-scope accuracy (no threshold),in-scope accepted,in-scope accepted and correct,out-of-scope rejected,balanced score
Reference network,0.550,0.100,0.075,0.96,0.518
TF-IDF + Logistic Regression,0.750,0.525,0.525,0.80,0.663
Nearest training message,0.725,0.350,0.325,0.84,0.582
"Reference network, no threshold",0.550,1.000,0.550,0.00,0.275


In [8]:
predicted, confidence = logistic_answer(in_test["message"].tolist())
errors = in_test.assign(predicted=predicted, confidence=confidence.round(2))
errors[errors["intent"] != errors["predicted"]].head(12)

,message,intent,predicted,confidence
13,Much appreciated,thanks,greeting,0.23
16,Introduce yourself,about,greeting,0.23
17,What is this assistant?,about,name,0.41
18,Who is this?,about,greeting,0.26
22,How do I refer to you?,name,createaccount,0.26
23,Name please,name,help,0.37
25,Is there anyone who can help?,help,greeting,0.38
30,I want to sign up for your service,createaccount,complaint,0.30
31,How can I register?,createaccount,help,0.34
37,Something went wrong and I am unhappy,complaint,greeting,0.23


The three models differ clearly on messages they have not seen:

- **Reference network:** 0.55 in-scope accuracy on the test messages, against 1.0 on its training messages. This is overfitting: with 33 training messages and 500 epochs it learns the exact words of the examples. Its confidence is also poor as a filter: it is very sure about everything, so without a threshold it accepts all the out-of-scope messages (rejected share 0.00) and answers them with a prepared reply.
- **TF-IDF + Logistic Regression** has the best in-scope accuracy (0.75) and the best balanced score (0.663). The nearest training message is close (0.725); with 40 test messages one message is 2.5 points, so these two are not clearly different, but both are clearly better than the network.
- **The threshold has a cost.** For the Logistic Regression it accepts only 52.5% of the in-scope messages (all of those accepted are correct, 52.5%), and rejects 80% of the out-of-scope messages. The other in-scope messages are sent to the language model, which is slower but still answers reasonably, while the messages accepted by mistake (20% of the out-of-scope ones) get a wrong prepared reply.

The errors of the Logistic Regression are mostly paraphrases that share no word with the training messages ("Much appreciated" for thanks, "How can I register?" for create account, "Who is this?" for about). A model that counts words cannot link words it has never seen together; this needs word meaning (word embeddings or a language model), and 33 messages are too few to learn it.

## The Agent: Prepared Replies and Language Model

The agent uses the model with the best balanced score. If the message is accepted, it answers with a random prepared reply of the intent. If not, it asks the language model, with a short instruction that keeps the answers brief. The language model is Llama 3.2 (3 billion parameters), run by Ollama on the CPU; its answers are not scored, they are only shown, together with the time they take.

In [11]:
#pip install ollama

In [12]:
import ollama

best_model_name = results.iloc[:3]["balanced score"].idxmax()
best_model, threshold = models[best_model_name], best_threshold[best_model_name]
replies = {intent["tag"]: intent["responses"] for intent in intents}
rng = np.random.default_rng(42)

def agent(message):
    start = time.time()
    intent, confidence = (a[0] for a in best_model([message]))
    if confidence >= threshold:
        return {"message": message, "route": f"prepared reply ({intent})", "answer": rng.choice(replies[intent]), "seconds": round(time.time() - start, 1)}
    answer = ollama.chat(model="llama3.2", messages=[{"role": "system", "content": "You are Joana, a friendly assistant of a website. Answer in one or two short sentences."},
                                                     {"role": "user", "content": message}], options={"temperature": 0.3, "num_predict": 60}, keep_alive="30m")
    return {"message": message, "route": "language model", "answer": answer["message"]["content"].strip(), "seconds": round(time.time() - start, 1)}

examples = ["Hello", "I need to open an account", "I want to complain about my order", "What is the capital of Italy?", "Tell me a joke", "Thanks, bye"]
pd.DataFrame([agent(m) for m in examples])

,message,route,answer,seconds
0,Hello,prepared reply (greeting),Hello,0.0
1,I need to open an account,prepared reply (createaccount),Just go to our web site and follow the guideli...,0.0
2,I want to complain about my order,language model,I'm so sorry to hear that you're not satisfied...,9.8
3,What is the capital of Italy?,language model,The capital of Italy is Rome.,0.9
4,Tell me a joke,prepared reply (help),Tell me your problem to assist you,0.0
5,"Thanks, bye",language model,It was nice chatting with you. Have a great day!,1.2


The demonstration shows the two routes. "Hello" and "I need to open an account" get a prepared reply immediately (0.0 seconds). The complaint, the question about Italy and "Thanks, bye" go to the language model, which answers correctly in 1 to 3 seconds on this CPU. There are also two mistakes of the classifier: "Tell me a joke" is accepted as a help request and gets an unrelated prepared reply, and "Thanks, bye" contains two intents and is not accepted by the classifier (the language model handles it well). A threshold reduces the wrong prepared replies but cannot remove them, as the evaluation showed.

## Conclusion

- The reference network memorizes its 33 training messages (accuracy 1.0) but reaches only 0.55 on new paraphrases of the same intents, and it answers every message, including the ones unrelated to its intents. The reference has no test data, so neither problem is visible in the article.
- TF-IDF with Logistic Regression is simpler, trains instantly and is better on new messages (0.75). With so few training messages a simple model is the more sensible choice.
- A confidence threshold lets the bot refuse messages that are not for it: 80% of the out-of-scope test messages are rejected, at the price of also sending about half of the in-scope messages to the language model.
- The agent is a fixed workflow (classifier, then prepared reply or local Llama 3.2). It covers the cases the prepared replies cannot, at 1 to 3 seconds per answer on a CPU. The language model answers are shown but not scored.
- Limitations: the test messages (40 in-scope, 25 out-of-scope) were written for this notebook and are few, and the threshold was chosen on only 65 development messages, so the scores are a rough guide. More training messages per intent would help more than a more complex model.